# Fase 1 — EDA: visualización de señales crudas

Primer paso de la Fase 1 (ver `FASES.md`): cargar una muestra de cada dataset (CODE-15%, SaMi-Trop, PTB-XL) y graficar las 12 derivaciones crudas, tal como viven en los HDF5 consolidados en Fase 0.

No se toca la señal (sin resamplear, sin normalizar): el objetivo es *ver* qué hay antes de decidir cómo preprocesarlo (eso es Fase 2).

In [ ]:
import sys
sys.path.insert(0, "../src")

import matplotlib.pyplot as plt
import numpy as np

from eda_utils import LEADS, cargar_metadata, cargar_senal

meta = cargar_metadata()
meta.groupby(["dataset", "confianza"]).agg(n=("record_id", "count"), positivos=("chagas_label", "sum"))

In [ ]:
def plot_12_derivaciones(row, senal: np.ndarray):
    fig, axes = plt.subplots(12, 1, figsize=(12, 14), sharex=True)
    t = np.arange(senal.shape[0]) / row["frecuencia"]
    for i, ax in enumerate(axes):
        ax.plot(t, senal[:, i], linewidth=0.6, color="#1f5fa8")
        ax.set_ylabel(LEADS[i], rotation=0, ha="right", va="center")
        ax.set_yticks([])
    axes[-1].set_xlabel("tiempo (s)")
    label = "Chagas+" if row["chagas_label"] else "Chagas-"
    fig.suptitle(
        f"{row['dataset']} — record_id={row['record_id']} — {label} ({row['confianza']}) — "
        f"{row['edad']:.0f} años, {row['sexo']}, {row['frecuencia']} Hz"
    )
    fig.tight_layout()
    return fig

## Una muestra por dataset

Semilla fija para poder reproducir el mismo registro entre corridas.

In [ ]:
rng_seed = 0
for dataset in ["code15", "samitrop", "ptbxl"]:
    row = meta[meta["dataset"] == dataset].sample(1, random_state=rng_seed).iloc[0]
    senal = cargar_senal(row)
    plot_12_derivaciones(row, senal)
    plt.show()

## Varias muestras del mismo dataset, para comparar

Cambiar `dataset` y `rng_seed` para explorar otros registros. Útil para tener una idea de la variabilidad dentro de un mismo dataset antes de mirar entre datasets.

In [ ]:
dataset = "code15"
n_muestras = 3

sub = meta[meta["dataset"] == dataset].sample(n_muestras, random_state=1)
for _, row in sub.iterrows():
    senal = cargar_senal(row)
    plot_12_derivaciones(row, senal)
    plt.show()